# Apriori vs. FP-Growth vs. PrefixSpan on the COVID-19 Spread Chain Binary Matrix

This notebook applies **Apriori, FP-Growth, and PrefixSpan** to
`Binary_Matrix_of_Spread_Chain.csv` in one place, so all three can be compared directly.
As before, the **only** preprocessing step is removing the fully empty rows in the file —
no other cleaning, filtering, or value changes are made anywhere in this notebook.

**Important note before you start — please read this one:** Apriori and FP-Growth are
*frequent itemset* algorithms: they only care which destination countries appear *together*
in a source country's row, not in what order. **PrefixSpan is a *sequential pattern* algorithm**
— it specifically looks for orderings (A happens, then later B happens). This dataset has **no
time or order information at all**: it's a binary presence/absence matrix, and the only "order"
the destination-country columns have is the alphabetical order they happen to be listed in the
file (Afghanistan, Albania, Algeria, ...). To run PrefixSpan at all, each source country's row is
turned into a sequence by listing its destination countries in that column order — but that order
is **alphabetical, not chronological or epidemiological**. Any "A occurs before B" pattern
PrefixSpan reports below reflects alphabetical position in the file, not a real transmission
order. This is disclosed clearly in Step 6 and in the comparison section, so the result isn't
mistaken for genuine sequence-of-spread evidence — that would require a dataset with actual
timestamps or travel-order data per country pair.

In [1]:
# Step 0 — Install the libraries all three algorithms come from
!pip install mlxtend prefixspan --quiet

  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done


**What this does:** `mlxtend` (Machine Learning Extensions) provides standard, tested
implementations of `apriori`, `fpgrowth`, and `association_rules`. `prefixspan` is a separate,
dedicated Python package implementing the PrefixSpan sequential pattern mining algorithm — it
isn't part of `mlxtend`, since sequence mining is a different family of algorithm from itemset
mining.

In [2]:
# Step 1 — Upload the CSV to this Colab session
from google.colab import files
uploaded = files.upload()   # choose Binary_Matrix_of_Spread_Chain.csv when prompted

filename = list(uploaded.keys())[0]
print("Uploaded file:", filename)

Saving Binary Matrix of Spread Chain.csv to Binary Matrix of Spread Chain.csv
Uploaded file: Binary Matrix of Spread Chain.csv


**What this does:** Colab runs on a remote machine with no access to your local disk, so
the file must be uploaded into the session's temporary storage first. `files.upload()` opens a
file picker; once you select the CSV it's saved into the notebook's working directory.

In [3]:
# Step 2 — Load the raw CSV
import pandas as pd

df = pd.read_csv(filename)
print("Shape as loaded:", df.shape)
df.head()

Shape as loaded: (42, 187)


,Source Country,Afghanistan,Albania,Algeria,Andorra,Angola,Antigua and Barbuda,Argentina,Armenia,Australia,...,Uruguay,Uzbekistan,Vatican City,Venezuela,Vietnam,West Bank and Gaza,Western Sahara,Yemen,Zambia,Zimbabwe
0,Italy,0,1,1,1,0,0,1,0,0,...,1,0,0,0,0,0,0,0,0,0
1,China,0,0,0,0,0,0,0,0,1,...,0,0,0,0,1,0,0,0,0,0
2,Iran,1,0,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
3,United Kingdom,0,0,0,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,1
4,France,0,0,0,0,0,0,0,0,0,...,0,1,0,0,0,0,0,0,1,0


**What this does:** reads the file into a pandas DataFrame with no modification —
every row and column is loaded exactly as it appears in the CSV.

In [4]:
# Step 3 — Preprocessing: remove empty rows only
# The file has trailing rows with no Source Country and no data in any column at all.
# This is the only preprocessing step performed: drop rows with nothing in them.
df = df.dropna(subset=['Source Country'])

print("Shape after removing empty rows:", df.shape)
df.head()

Shape after removing empty rows: (42, 187)


,Source Country,Afghanistan,Albania,Algeria,Andorra,Angola,Antigua and Barbuda,Argentina,Armenia,Australia,...,Uruguay,Uzbekistan,Vatican City,Venezuela,Vietnam,West Bank and Gaza,Western Sahara,Yemen,Zambia,Zimbabwe
0,Italy,0,1,1,1,0,0,1,0,0,...,1,0,0,0,0,0,0,0,0,0
1,China,0,0,0,0,0,0,0,0,1,...,0,0,0,0,1,0,0,0,0,0
2,Iran,1,0,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
3,United Kingdom,0,0,0,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,1
4,France,0,0,0,0,0,0,0,0,0,...,0,1,0,0,0,0,0,0,1,0


**What this does:** drops rows where `Source Country` is empty. In this file, those rows
have no values in any column, so this removes rows with no data rather than changing, filtering,
or reinterpreting any actual `0`/`1` value. No other step is applied — every remaining cell keeps
its original value, and this same cleaned table feeds **all three** algorithms below, so none of
them gets an advantage from different input data.

In [5]:
# Step 4 — Format for Apriori and FP-Growth
# Both need 'Source Country' set aside as the transaction label (not treated as an item),
# and the item columns as True/False. Format conversion only -- no value changes.
transactions = df.set_index('Source Country').astype(bool)

print("Transactions shape (source countries x destination countries):", transactions.shape)
transactions.head()

Transactions shape (source countries x destination countries): (42, 186)


,Afghanistan,Albania,Algeria,Andorra,Angola,Antigua and Barbuda,Argentina,Armenia,Australia,Austria,...,Uruguay,Uzbekistan,Vatican City,Venezuela,Vietnam,West Bank and Gaza,Western Sahara,Yemen,Zambia,Zimbabwe
Source Country,,,,,,,,,,,,,,,,,,,,,
Italy,False,True,True,True,False,False,True,False,False,True,...,True,False,False,False,False,False,False,False,False,False
China,False,False,False,False,False,False,False,False,True,False,...,False,False,False,False,True,False,False,False,False,False
Iran,True,False,False,False,False,False,False,True,False,False,...,False,False,False,False,False,False,False,False,False,False
United Kingdom,False,False,False,False,False,True,False,False,False,False,...,False,False,False,False,False,False,False,False,False,True
France,False,False,False,False,False,False,False,False,False,False,...,False,True,False,False,False,False,False,False,True,False


**What this does:** `Source Country` becomes the row label (the transaction ID), and the
186 remaining columns (destination countries) become items. Each cell keeps the same 0/1
information from the file, just typed as `True`/`False`, which both `apriori()` and `fpgrowth()`
require internally.

In [6]:
# Step 5 — Run Apriori
import time
from mlxtend.frequent_patterns import apriori

start = time.time()
frequent_itemsets_apriori = apriori(transactions, min_support=0.04, use_colnames=True)
time_apriori = time.time() - start

frequent_itemsets_apriori = frequent_itemsets_apriori.sort_values(by='support', ascending=False)
print(f"Apriori found {len(frequent_itemsets_apriori)} frequent itemsets in {time_apriori:.4f} seconds")
frequent_itemsets_apriori.head(10)

Apriori found 25 frequent itemsets in 0.0093 seconds


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,support,itemsets
20,0.095238,(Togo)
16,0.071429,(Peru)
22,0.071429,(West Bank and Gaza)
15,0.071429,(Niger)
4,0.047619,(Cyprus)
1,0.047619,(Benin)
2,0.047619,(Botswana)
3,0.047619,(Burundi)
0,0.047619,(Bahrain)
8,0.047619,(Honduras)


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


**What this does:** Apriori repeatedly scans the transaction table, generating candidate
itemsets of increasing size and directly checking each one's support against the data, pruning
any branch built from an already-infrequent itemset.

- `min_support=0.04`: with only 42 transactions, an itemset appearing in just 2 rows already has
  support = 2/42 ≈ 0.048, so this threshold needs to stay low to find anything at all.

In [7]:
# Step 6 — Run FP-Growth
from mlxtend.frequent_patterns import fpgrowth

start = time.time()
frequent_itemsets_fpgrowth = fpgrowth(transactions, min_support=0.04, use_colnames=True)
time_fpgrowth = time.time() - start

frequent_itemsets_fpgrowth = frequent_itemsets_fpgrowth.sort_values(by='support', ascending=False)
print(f"FP-Growth found {len(frequent_itemsets_fpgrowth)} frequent itemsets in {time_fpgrowth:.4f} seconds")
frequent_itemsets_fpgrowth.head(10)

FP-Growth found 25 frequent itemsets in 0.0091 seconds


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

,support,itemsets
12,0.095238,(Togo)
20,0.071429,(Niger)
23,0.071429,(West Bank and Gaza)
13,0.071429,(Peru)
4,0.047619,(Saudi Arabia)
1,0.047619,(Saint Kitts and Nevis)
2,0.047619,(Portugal)
3,0.047619,(Cyprus)
0,0.047619,(Ukraine)
8,0.047619,(Zimbabwe)


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


**What this does:** FP-Growth scans the data twice, builds a compressed tree structure
(the "FP-tree") encoding all itemset/frequency information, and mines frequent itemsets directly
from that tree — without generating and testing candidates one at a time the way Apriori does.
It runs on the exact same `transactions` table and `min_support`, so its result is directly
comparable to Apriori's.

In [8]:
# Step 7 — Build sequences and run PrefixSpan
# PrefixSpan needs a "sequence database": a list of sequences, each sequence being an ordered
# list of items. This dataset has no timestamps, so each source country's row is converted into
# a sequence by listing its destination countries in the column order they appear in the file
# (alphabetical order -- see the note at the top of this notebook). No value is changed or
# filtered -- this only re-expresses the same 0/1 row as a list of the items that were 1.
from prefixspan import PrefixSpan

item_columns = [c for c in df.columns if c != 'Source Country']

sequences = []
for _, row in df.iterrows():
    seq = [country for country in item_columns if row[country] == 1]
    sequences.append(seq)

print("Number of sequences:", len(sequences))
print("Example sequence (first source country):", sequences[0][:10], "...")

min_count = 2  # a pattern must appear in at least 2 of the 42 sequences (support ~= 0.048), matching min_support=0.04 used above

start = time.time()
ps = PrefixSpan(sequences)
patterns = ps.frequent(min_count)
time_prefixspan = time.time() - start

patterns_df = pd.DataFrame(patterns, columns=['count', 'pattern'])
patterns_df['support'] = patterns_df['count'] / len(sequences)
patterns_df = patterns_df.sort_values(by='support', ascending=False)

print(f"PrefixSpan found {len(patterns_df)} frequent sequential patterns in {time_prefixspan:.4f} seconds")
patterns_df.head(10)

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

Number of sequences: 42
Example sequence (first source country): ['Albania', 'Algeria', 'Andorra', 'Argentina', 'Austria', 'Bangladesh', 'Bolivia', 'Bosnia and Herzegovina', 'Brazil', 'Central African Republic'] ...
PrefixSpan found 25 frequent sequential patterns in 0.0006 seconds


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,count,pattern,support
14,4,[Togo],0.095238
24,3,[West Bank and Gaza],0.071429
21,3,[Niger],0.071429
13,3,[Peru],0.071429
4,2,[Bahrain],0.047619
1,2,[Portugal],0.047619
2,2,[Saint Kitts and Nevis],0.047619
3,2,[Ukraine],0.047619
0,2,[Cyprus],0.047619
8,2,[Botswana],0.047619


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


**What this does:** `PrefixSpan(sequences)` builds the search structure over the sequence
database, and `.frequent(min_count)` recursively grows frequent subsequences prefix by prefix —
finding, at each step, the items that can extend an existing frequent subsequence while staying
above `min_count`, and stopping when no extension stays frequent. `min_count=2` sequences is used
so the threshold matches the `min_support=0.04` (~2/42) used for Apriori and FP-Growth above,
keeping the three comparable.

A pattern with more than one country, e.g. `['Kenya', 'Myanmar']`, means: in every sequence
containing both, Kenya appears in a column position before Myanmar's. As the note at the top
explains, "before" here is alphabetical column order, not real chronological order — so this
tells you Kenya and Myanmar co-occur, but not that one genuinely preceded the other in time.

In [9]:
# Step 8 — Generate association rules for Apriori and FP-Growth
from mlxtend.frequent_patterns import association_rules

rules_apriori = association_rules(frequent_itemsets_apriori, metric="confidence", min_threshold=0.5)
rules_apriori = rules_apriori.sort_values(by='lift', ascending=False)

rules_fpgrowth = association_rules(frequent_itemsets_fpgrowth, metric="confidence", min_threshold=0.5)
rules_fpgrowth = rules_fpgrowth.sort_values(by='lift', ascending=False)

print(f"Apriori rules: {len(rules_apriori)}   FP-Growth rules: {len(rules_fpgrowth)}")
rules_apriori[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head(10)

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

Apriori rules: 2   FP-Growth rules: 2


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

,antecedents,consequents,support,confidence,lift
0,(Kenya),(Myanmar),0.047619,1.0,21.0
1,(Myanmar),(Kenya),0.047619,1.0,21.0


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


**What this does:** turns Apriori's and FP-Growth's frequent itemsets into if-then rules
of the form **{A} -> {B}**, each with support, confidence, and lift (as in the earlier
single-algorithm notebooks). PrefixSpan's package doesn't have an equivalent built-in
rule-generation step — its native output is the ordered patterns themselves, which is why the
comparison below compares patterns/itemsets directly rather than rules for all three.

In [10]:
# Step 9 — Compare all three algorithms
print("="*65)
print("COMPARISON: Apriori vs. FP-Growth vs. PrefixSpan")
print("="*65)

print(f"\nRuntime:")
print(f"  Apriori:     {time_apriori:.4f} seconds")
print(f"  FP-Growth:   {time_fpgrowth:.4f} seconds")
print(f"  PrefixSpan:  {time_prefixspan:.4f} seconds")

print(f"\nFrequent patterns found:")
print(f"  Apriori itemsets:      {len(frequent_itemsets_apriori)}")
print(f"  FP-Growth itemsets:    {len(frequent_itemsets_fpgrowth)}")
print(f"  PrefixSpan sequences:  {len(patterns_df)}")

# Apriori and FP-Growth solve the identical problem, so they must find the same itemsets
itemsets_apriori = set(frozenset(s) for s in frequent_itemsets_apriori['itemsets'])
itemsets_fpgrowth = set(frozenset(s) for s in frequent_itemsets_fpgrowth['itemsets'])
print(f"\nApriori and FP-Growth found identical itemsets: {itemsets_apriori == itemsets_fpgrowth}")

# PrefixSpan's multi-country patterns, read as unordered sets, should match the same pairs/groups
# found by Apriori and FP-Growth -- since with one itemset per sequence there's no extra ordering
# information for PrefixSpan to exploit beyond plain co-occurrence.
prefixspan_itemsets = set(frozenset(p) for _, p in patterns if len(p) > 0)
overlap = prefixspan_itemsets & itemsets_apriori
print(f"PrefixSpan patterns that also appear as Apriori/FP-Growth itemsets (as unordered sets): {len(overlap)} / {len(prefixspan_itemsets)}")

COMPARISON: Apriori vs. FP-Growth vs. PrefixSpan

Runtime:
  Apriori:     0.0093 seconds
  FP-Growth:   0.0091 seconds
  PrefixSpan:  0.0006 seconds

Frequent patterns found:
  Apriori itemsets:      25
  FP-Growth itemsets:    25
  PrefixSpan sequences:  25

Apriori and FP-Growth found identical itemsets: True
PrefixSpan patterns that also appear as Apriori/FP-Growth itemsets (as unordered sets): 25 / 25


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

**What this does, and what to expect:**

| | Apriori | FP-Growth | PrefixSpan |
|---|---|---|---|
| Mines | Frequent itemsets (unordered) | Frequent itemsets (unordered) | Frequent **sequences** (ordered) |
| Approach | Generates and tests candidate itemsets level by level | Builds one compressed tree (FP-tree), mines it directly | Recursively grows frequent subsequences via projected databases |
| Data scans | Multiple (one per itemset size) | Two, total | One initial pass, then recursive on projections |
| Needs order/time info? | No | No | Yes — the algorithm's whole purpose is finding orderings |
| Result on *this* dataset | Frequent country co-occurrences | Same as Apriori | Same country co-occurrences, "ordered" only by alphabetical column position |

Apriori and FP-Growth should show `True` for identical itemsets — that's expected, since they're
two search strategies for the same well-defined problem. PrefixSpan's multi-item patterns, once
you ignore the (alphabetical, not real) ordering, should substantially overlap with the same
country groups Apriori and FP-Growth found — because with only one itemset per sequence in this
data, PrefixSpan has no genuine time-ordering to exploit beyond plain co-occurrence, so it
collapses close to the same answer as the itemset algorithms rather than showing its real
strength.

**The honest takeaway for the paper:** all three algorithms agree on *which* countries co-occur
in spread chains, which is a solid three-way cross-validation of that finding. But PrefixSpan is
not being used for what it's actually designed for here — meaningful sequential mining on this
data would need a version of the dataset with real time-ordering (e.g. first-case dates per
country, mentioned in your project's international-spread follow-up work) so "A then B" reflects
an actual chronological chain rather than the alphabet.

In [11]:
# Step 10 — Show the strongest results from each algorithm side by side
print("Top Apriori rules:\n")
for _, row in rules_apriori.sort_values(['lift','confidence'], ascending=False).head(5).iterrows():
    a = ', '.join(list(row['antecedents'])); c = ', '.join(list(row['consequents']))
    print(f"  {a} -> {c}   (support={row['support']:.3f}, confidence={row['confidence']:.3f}, lift={row['lift']:.3f})")

print("\nTop FP-Growth rules:\n")
for _, row in rules_fpgrowth.sort_values(['lift','confidence'], ascending=False).head(5).iterrows():
    a = ', '.join(list(row['antecedents'])); c = ', '.join(list(row['consequents']))
    print(f"  {a} -> {c}   (support={row['support']:.3f}, confidence={row['confidence']:.3f}, lift={row['lift']:.3f})")

print("\nTop PrefixSpan sequential patterns (length > 1):\n")
multi_patterns = patterns_df[patterns_df['pattern'].apply(len) > 1].head(5)
for _, row in multi_patterns.iterrows():
    print(f"  {' -> '.join(row['pattern'])}   (support={row['support']:.3f}, count={row['count']})")

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

Top Apriori rules:

  Kenya -> Myanmar   (support=0.048, confidence=1.000, lift=21.000)
  Myanmar -> Kenya   (support=0.048, confidence=1.000, lift=21.000)

Top FP-Growth rules:

  Kenya -> Myanmar   (support=0.048, confidence=1.000, lift=21.000)
  Myanmar -> Kenya   (support=0.048, confidence=1.000, lift=21.000)

Top PrefixSpan sequential patterns (length > 1):

  Kenya -> Myanmar   (support=0.048, count=2)


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

**What this does:** prints the top results from all three algorithms together for a quick
read, without needing to scroll through the full DataFrames above. As noted throughout, the
PrefixSpan `->` here denotes alphabetical column order, not a confirmed real-world sequence.

In [12]:
# Step 11 (optional) — Save all results for use in the paper
frequent_itemsets_apriori.to_csv('apriori_frequent_itemsets.csv', index=False)
rules_apriori.to_csv('apriori_association_rules.csv', index=False)
frequent_itemsets_fpgrowth.to_csv('fpgrowth_frequent_itemsets.csv', index=False)
rules_fpgrowth.to_csv('fpgrowth_association_rules.csv', index=False)
patterns_df.to_csv('prefixspan_patterns.csv', index=False)

from google.colab import files
files.download('apriori_frequent_itemsets.csv')
files.download('apriori_association_rules.csv')
files.download('fpgrowth_frequent_itemsets.csv')
files.download('fpgrowth_association_rules.csv')
files.download('prefixspan_patterns.csv')

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

<IPython.core.display.Javascript object>

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


<IPython.core.display.Javascript object>

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


...

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

<IPython.core.display.Javascript object>

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


<IPython.core.display.Javascript object>

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

**What this does:** exports the frequent itemsets and rules from Apriori and FP-Growth,
plus PrefixSpan's frequent patterns, to separate CSV files and downloads them all — useful for
building a three-way comparison table/figure in the paper.